# 句子的embedding

In [5]:
import torch
from transformers import AutoTokenizer, AutoModel

sentences = [
    'I took my dog for a walk',
    'Today is going to rain',
    'I took my cat for a walk',
]
print(f'sentences: {sentences}')

model_checkpoint = 'sentence-transformers/all-MiniLM-L6-v2'
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModel.from_pretrained(model_checkpoint)

encoded_input = tokenizer(
    sentences, padding=True, truncation=True, return_tensors="pt"
)
print(f'encoded_input["input_ids"]: {encoded_input["input_ids"]}')
decoded_input = tokenizer.batch_decode(encoded_input["input_ids"])
print(f'decoded_input: {decoded_input}')

with torch.no_grad():
    model_output = model(**encoded_input)

token_embeddings = model_output.last_hidden_state
print(f'token_embeddings.shape: {token_embeddings.shape}')


sentences: ['I took my dog for a walk', 'Today is going to rain', 'I took my cat for a walk']


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


encoded_input["input_ids"]: tensor([[ 101, 1045, 2165, 2026, 3899, 2005, 1037, 3328,  102],
        [ 101, 2651, 2003, 2183, 2000, 4542,  102,    0,    0],
        [ 101, 1045, 2165, 2026, 4937, 2005, 1037, 3328,  102]])
decoded_input: ['[CLS] i took my dog for a walk [SEP]', '[CLS] today is going to rain [SEP] [PAD] [PAD]', '[CLS] i took my cat for a walk [SEP]']
token_embeddings.shape: torch.Size([3, 9, 384])


# 池化

In [36]:
import torch
import torch.nn.functional as F

def mean_pooling(model_output, attention_mask):
    # model_output.last_hidden_state: shape [batch, seq_len, hidden_dim]
    # 每个 token 对应一个 384 维向量，我们要把一句话的所有 token 向量"平均"成一个句向量
    token_embeddings = model_output.last_hidden_state
    print(f'token_embeddings.shape: {token_embeddings.shape}')
    # shape: [batch=3, seq_len=9, hidden_dim=384]

    # attention_mask: shape [batch, seq_len]，1 = 真实 token，0 = [PAD]
    print(f'attention_mask:\n{encoded_input["attention_mask"]}')

    # unsqueeze(-1): [batch, seq_len] → [batch, seq_len, 1]
    # expand(...): 广播到 [batch, seq_len, hidden_dim]，让每一维 embedding 都乘上对应的 mask
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    print(f'input_mask_expanded.shape: {input_mask_expanded.shape}')
    # shape: [3, 9, 384]，PAD 位置全为 0，真实 token 位置全为 1

    # token_embeddings * input_mask_expanded：把 PAD token 的向量清零
    masked = token_embeddings * input_mask_expanded
    print(f'masked embeddings (PAD zeroed out), shape: {masked.shape}')

    # sum(dim=1)：在 seq_len 维度求和 → [batch, hidden_dim]
    summed = torch.sum(masked, 1)

    # input_mask_expanded.sum(1)：每个样本有效 token 数，shape [batch, hidden_dim]
    # clamp(min=1e-9)：防止除以 0
    counts = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
    print(f'有效 token 数 (per sentence): {counts[:, 0]}\n')
    # 句子1: 9个token (含[CLS][SEP]), 句子2: 7个token (其余为[PAD]), 句子3: 9个token

    # 用有效 token 数做加权平均
    return summed / counts

# ── 第一步：Mean Pooling ────────────────────────────────────────────────
# 目标：把 [3, 9, 384] 的 token 向量压缩成 [3, 384] 的句向量
# 做法：对每个句子的真实 token 向量求均值（忽略 [PAD]）
sentence_embeddings = mean_pooling(model_output, encoded_input['attention_mask'])
print(f'\n[Mean Pooling 后] sentence_embeddings.shape: {sentence_embeddings.shape}')
# shape: [3, 384]，每句话得到一个 384 维向量

# ── 第二步：L2 归一化 ───────────────────────────────────────────────────
# 将每个句向量归一化到单位球面（模长=1），使余弦相似度 = 点积，计算更稳定
before_norm = sentence_embeddings[0].norm().item()
sentence_embeddings = F.normalize(sentence_embeddings, p=2, dim=1)
after_norm = sentence_embeddings[0].norm().item()
print(f'归一化前第0句向量模长: {before_norm:.4f}')
print(f'归一化后第0句向量模长: {after_norm:.4f}')  # ≈ 1.0
print(f'归一化后第1句向量模长: {sentence_embeddings[1].norm().item():.4f}')
print(f'归一化后第2句向量模长: {sentence_embeddings[2].norm().item():.4f}')


# ── 第三步：计算句子间余弦相似度 ────────────────────────────────────────
# 归一化后：cos_sim(a, b) = a · b（点积）
cos_sim = sentence_embeddings @ sentence_embeddings.T
print(f'\n余弦相似度矩阵 (3×3):\n{cos_sim.numpy().round(4)}')
print()
for i, s1 in enumerate(sentences):
    for j, s2 in enumerate(sentences):
        if j > i:
            sim = cos_sim[i, j].item()
            print(f'  "{s1}"  ↔  "{s2}"')
            print(f'  相似度: {sim:.4f}\n')
# 预期：dog/cat 两句（语义近）相似度 >> dog/cat 与 rain（语义远）


token_embeddings.shape: torch.Size([3, 9, 384])
attention_mask:
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1]])
input_mask_expanded.shape: torch.Size([3, 9, 384])
masked embeddings (PAD zeroed out), shape: torch.Size([3, 9, 384])
有效 token 数 (per sentence): tensor([9., 7., 9.])


[Mean Pooling 后] sentence_embeddings.shape: torch.Size([3, 384])
归一化前第0句向量模长: 6.1770
归一化后第0句向量模长: 1.0000
归一化后第1句向量模长: 1.0000
归一化后第2句向量模长: 1.0000

余弦相似度矩阵 (3×3):
[[1.     0.1702 0.8291]
 [0.1702 1.     0.174 ]
 [0.8291 0.174  1.    ]]

  "I took my dog for a walk"  ↔  "Today is going to rain"
  相似度: 0.1702

  "I took my dog for a walk"  ↔  "I took my cat for a walk"
  相似度: 0.8291

  "Today is going to rain"  ↔  "I took my cat for a walk"
  相似度: 0.1740



# 语义搜索

In [59]:
from datasets import load_dataset

squad = load_dataset("squad", split="validation[:100]")

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
model = model.to(device)

def get_embeddings(text_list):
    print(f'text_list: {text_list}')
    encoded_input = tokenizer(text_list, padding=True, truncation=True, return_tensors="pt")
    encoded_input = {k: v.to(device) for k, v in encoded_input.items()}
    with torch.no_grad():
        model_output = model(**encoded_input)
    return mean_pooling(model_output, encoded_input["attention_mask"])

    
squad_context_embeddings = get_embeddings(list(squad.select_columns(["context"])['context']))

squad_with_embeddings = squad.map(lambda x: {"embeddings": get_embeddings(x["context"]).cpu().numpy()[0]})

text_list: ['Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super Bowl title. The game was played on February 7, 2016, at Levi\'s Stadium in the San Francisco Bay Area at Santa Clara, California. As this was the 50th Super Bowl, the league emphasized the "golden anniversary" with various gold-themed initiatives, as well as temporarily suspending the tradition of naming each Super Bowl game with Roman numerals (under which the game would have been known as "Super Bowl L"), so that the logo could prominently feature the Arabic numerals 50.', 'Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the Na

In [60]:
import numpy as np
np.array(squad_with_embeddings['embeddings'][:3])[:,:5], squad_context_embeddings[:3, :5]

(array([[ 0.00935947,  0.08776328,  0.03180885, -0.21667053, -0.09006277],
        [ 0.00935947,  0.08776328,  0.03180885, -0.21667053, -0.09006277],
        [ 0.00935947,  0.08776328,  0.03180885, -0.21667053, -0.09006277]]),
 tensor([[ 0.0094,  0.0878,  0.0318, -0.2167, -0.0901],
         [ 0.0094,  0.0878,  0.0318, -0.2167, -0.0901],
         [ 0.0094,  0.0878,  0.0318, -0.2167, -0.0901]], device='mps:0'))

# FAISS索引

In [ ]:
! pip install faiss-gpu

In [ ]:
squad_with_embeddings.add_faiss_index(column="embeddings")

question = 'Who headlined the halftime show for Super Bowl 50?'
question_embeddings = get_embeddings([question]).cpu().detach().numpy()
print(f'question_embeddings.shape: {question_embeddings.shape}')

scores, samples = squad_with_embeddings.get_nearest_examples(
    "embeddings", question_embeddings, k=3
)